# Multi-Agent Deepfake Detection: Data Download & Feature Extraction

This notebook downloads the ASVspoof5 dataset from Kaggle, samples 40% of the files for training and testing, and extracts tabular spectral and prosodic features using multi-processing. The results are saved to Google Drive.

In [ ]:
!pip install kaggle librosa soundfile praat-parselmouth xgboost tqdm pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/40_PER_22_Data')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using Google Drive directory: {DRIVE_DIR}")

## 1. Download Dataset

In [ ]:
import os

# Set your Kaggle credentials here or upload kaggle.json
os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY"

DATA_DIR = Path('/content/data/asvspoof5')

if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading ASVspoof5 from Kaggle...")
    !kaggle datasets download -d aniket202411001/asvspoof5-flac --unzip -p {DATA_DIR}
else:
    print("Dataset already exists.")

## 2. Sample 40% of the Dataset

In [ ]:
import random
import shutil
from tqdm.auto import tqdm

SAMPLE_FRACTION = 0.40
random.seed(42)

splits = ['train', 'test']
classes = ['bonafide', 'spoof']

sampled_files = []

for split in splits:
    for cls in classes:
        # The actual Kaggle paths based on the structure
        src_dir = DATA_DIR / "ASV_Main" / split / cls
        dest_dir = DRIVE_DIR / split / cls
        dest_dir.mkdir(parents=True, exist_ok=True)
        
        if src_dir.exists():
            files = list(src_dir.glob('*.flac'))
            n_sample = int(len(files) * SAMPLE_FRACTION)
            sampled = random.sample(files, n_sample)
            
            print(f"Copying {n_sample} files for {split}/{cls}...")
            for f in tqdm(sampled, leave=False):
                dest_path = dest_dir / f.name
                if not dest_path.exists():
                    shutil.copy2(f, dest_path)
                sampled_files.append((dest_path, 0 if cls == 'bonafide' else 1, split))
        else:
            print(f"Warning: {src_dir} not found!")

print(f"Total sampled files copied to drive: {len(sampled_files)}")

## 3. Extract Features (Optimized Combined Extraction)
**IMPORTANT:** Ensure you upload the latest `spectral_feature_extractor.py` and `prosodic_feature_extractor.py` to your Colab workspace before running the next cell. Alternatively, clone your github repo.

In [ ]:
import pandas as pd
import numpy as np
import librosa
import concurrent.futures
import multiprocessing
from spectral_feature_extractor import extract_spectral_row
from prosodic_feature_extractor import extract_prosodic_row

def process_combined(file_info):
    path, label, split = file_info
    try:
        # 1. LOAD ONCE! This saves 50% of disk I/O time.
        y, sr = librosa.load(str(path), sr=16000, mono=True)
        
        # 2. Extract both from the pre-loaded waveform
        spectral_feats = extract_spectral_row(y, sr=sr)
        prosodic_feats = extract_prosodic_row(y, sr=sr)
        
        spectral_feats['filename'] = path.name
        spectral_feats['label'] = label
        spectral_feats['split'] = split
        
        prosodic_feats['filename'] = path.name
        prosodic_feats['label'] = label
        prosodic_feats['split'] = split
        
        return (spectral_feats, prosodic_feats)
    except Exception as e:
        # print(f"Error processing {path.name}: {e}")
        return None

In [ ]:
def extract_all_features():
    spectral_results = []
    prosodic_results = []
    
    workers = max(4, multiprocessing.cpu_count())
    print(f"Using {workers} parallel processes for COMBINED extraction...")
    
    # We use executor.map with chunksize for massive IPC speedup
    with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
        # Using map with chunksize > 1 is much faster than submit for large iterables
        results = list(tqdm(executor.map(process_combined, sampled_files, chunksize=50), total=len(sampled_files)))
        
    for res in results:
        if res is not None:
            spectral_results.append(res[0])
            prosodic_results.append(res[1])
                
    s_df = pd.DataFrame(spectral_results)
    s_df.to_csv(DRIVE_DIR / 'spectral_features.csv', index=False)
    print(f"Saved {len(s_df)} rows to spectral_features.csv")
    
    p_df = pd.DataFrame(prosodic_results)
    p_df.to_csv(DRIVE_DIR / 'prosodic_features.csv', index=False)
    print(f"Saved {len(p_df)} rows to prosodic_features.csv")
    
    return s_df, p_df

In [ ]:
print("Extracting Combined Features...")
spectral_df, prosodic_df = extract_all_features()

print("Spectral preview:")
display(spectral_df.head(2))
print("Prosodic preview:")
display(prosodic_df.head(2))